In [1]:
# Complex RBAC Setup — run in Jupyter
import requests
import json

POLARIS_URL = 'http://192.168.139.2:8181'
REALM = 'POLARIS'

def get_token():
    r = requests.post(
        f'{POLARIS_URL}/api/catalog/v1/oauth/tokens',
        headers={'Polaris-Realm': REALM},
        data={
            'grant_type': 'client_credentials',
            'client_id': 'root',
            'client_secret': 'polaris-secret',
            'scope': 'PRINCIPAL_ROLE:ALL'
        }
    )
    return r.json()['access_token']

def h(token):
    return {
        'Authorization': f'Bearer {token}',
        'Polaris-Realm': REALM,
        'Content-Type': 'application/json'
    }

def post(url, token, body):
    r = requests.post(url, headers=h(token), json=body)
    return r.status_code, r.json() if r.text else {}

def put(url, token, body):
    r = requests.put(url, headers=h(token), json=body)
    return r.status_code, r.json() if r.text else {}

TOKEN = get_token()
BASE = f'{POLARIS_URL}/api/management/v1'
print('✅ Connected')

# ============================================================
# RBAC Structure to create:
#
# Principals:
#   alice         → data-engineer
#   bob           → data-analyst
#   carol         → ml-engineer
#   trino-service → data-analyst
#   etl-service   → data-engineer
#
# Principal Roles:
#   data-engineer  → full write on analytics-catalog
#   data-analyst   → read-only on analytics-catalog
#   ml-engineer    → write on ml-catalog only
#   ops-admin      → admin on all catalogs
#
# Catalogs:
#   analytics-catalog → s3a://data-catalog-bucket/analytics/
#   ml-catalog        → s3a://data-catalog-bucket/ml/
#
# Catalog Roles per catalog:
#   analytics-catalog:
#     engineer-role  → CATALOG_MANAGE_CONTENT
#     analyst-role   → TABLE_READ_DATA, NAMESPACE_LIST
#     admin-role     → CATALOG_MANAGE_ACCESS
#   ml-catalog:
#     ml-role        → CATALOG_MANAGE_CONTENT
#     admin-role     → CATALOG_MANAGE_ACCESS
# ============================================================

# Step 1 — Create Principal Roles
print('\n📋 Creating Principal Roles...')
principal_roles = ['data-engineer', 'data-analyst', 'ml-engineer', 'ops-admin']
for role in principal_roles:
    status, resp = post(f'{BASE}/principal-roles', TOKEN,
                        {'principalRole': {'name': role}})
    print(f'  {"✅" if status == 201 else "⚠️ "} principal-role: {role} ({status})')

# Step 2 — Create Principals
print('\n👤 Creating Principals...')
principals = [
    ('alice',         'data-engineer'),
    ('bob',           'data-analyst'),
    ('carol',         'ml-engineer'),
    ('trino-service', 'data-analyst'),
    ('etl-service',   'data-engineer'),
]
principal_credentials = {}
for pname, _ in principals:
    status, resp = post(f'{BASE}/principals', TOKEN,
                        {'name': pname, 'type': 'SERVICE'})
    if status == 201:
        creds = resp.get('credentials', {})
        principal_credentials[pname] = creds
        print(f'  ✅ principal: {pname} — clientId: {creds.get("clientId", "N/A")}')
    else:
        print(f'  ⚠️  principal: {pname} ({status}) — {resp}')

# Step 3 — Assign Principal Roles to Principals
print('\n🔗 Assigning Principal Roles...')
for pname, role in principals:
    status, _ = put(
        f'{BASE}/principals/{pname}/principal-roles',
        TOKEN,
        {'principalRole': {'name': role}}
    )
    print(f'  {"✅" if status in [200,201] else "⚠️ "} {pname} → {role} ({status})')

# Also assign ops-admin to alice (multiple roles)
status, _ = put(
    f'{BASE}/principals/alice/principal-roles',
    TOKEN,
    {'principalRole': {'name': 'ops-admin'}}
)
print(f'  {"✅" if status in [200,201] else "⚠️ "} alice → ops-admin ({status})')

# Step 4 — Create Catalogs
print('\n📚 Creating Catalogs...')
catalogs = [
    {
        'name': 'analytics-catalog',
        'location': 's3a://data-catalog-bucket/analytics'
    },
    {
        'name': 'ml-catalog',
        'location': 's3a://data-catalog-bucket/ml'
    }
]
for c in catalogs:
    status, resp = post(f'{BASE}/catalogs', TOKEN, {
        'catalog': {
            'name': c['name'],
            'type': 'INTERNAL',
            'properties': {
                'default-base-location': c['location'],
                's3.endpoint': 'http://benchmarks-minio.datahub-hynix.svc.cluster.local:9000',
                's3.path-style-access': 'true'
            },
            'storageConfigInfo': {
                'storageType': 'S3',
                'allowedLocations': ['s3a://data-catalog-bucket/'],
                'roleArn': 'arn:aws:iam::123456789012:role/dummy',
                'pathStyleAccess': True
            }
        }
    })
    print(f'  {"✅" if status == 201 else "⚠️ "} catalog: {c["name"]} ({status})')

# Step 5 — Create Catalog Roles
print('\n🔑 Creating Catalog Roles...')
catalog_roles = {
    'analytics-catalog': ['engineer-role', 'analyst-role', 'admin-role'],
    'ml-catalog':        ['ml-role', 'admin-role']
}
for cname, roles in catalog_roles.items():
    for role in roles:
        status, _ = post(
            f'{BASE}/catalogs/{cname}/catalog-roles',
            TOKEN,
            {'catalogRole': {'name': role}}
        )
        print(f'  {"✅" if status == 201 else "⚠️ "} {cname}/{role} ({status})')

# Step 6 — Grant Privileges to Catalog Roles
print('\n⚡ Granting Privileges...')
grants = [
    # analytics-catalog
    ('analytics-catalog', 'engineer-role', 'catalog', 'CATALOG_MANAGE_CONTENT'),
    ('analytics-catalog', 'analyst-role',  'catalog', 'CATALOG_READ_PROPERTIES'),
    ('analytics-catalog', 'analyst-role',  'catalog', 'TABLE_READ_DATA'),
    ('analytics-catalog', 'admin-role',    'catalog', 'CATALOG_MANAGE_ACCESS'),
    ('analytics-catalog', 'admin-role',    'catalog', 'CATALOG_MANAGE_CONTENT'),
    # ml-catalog
    ('ml-catalog',        'ml-role',       'catalog', 'CATALOG_MANAGE_CONTENT'),
    ('ml-catalog',        'admin-role',    'catalog', 'CATALOG_MANAGE_ACCESS'),
    ('ml-catalog',        'admin-role',    'catalog', 'CATALOG_MANAGE_CONTENT'),
]
for cname, role, gtype, priv in grants:
    status, _ = put(
        f'{BASE}/catalogs/{cname}/catalog-roles/{role}/grants',
        TOKEN,
        {'grant': {'type': gtype, 'privilege': priv}}
    )
    print(f'  {"✅" if status in [200,201] else "⚠️ "} {cname}/{role} → {priv} ({status})')

# Step 7 — Assign Catalog Roles to Principal Roles
print('\n🔗 Assigning Catalog Roles to Principal Roles...')
assignments = [
    ('data-engineer', 'analytics-catalog', 'engineer-role'),
    ('data-analyst',  'analytics-catalog', 'analyst-role'),
    ('ml-engineer',   'ml-catalog',        'ml-role'),
    ('ops-admin',     'analytics-catalog', 'admin-role'),
    ('ops-admin',     'ml-catalog',        'admin-role'),
]
for pr, catalog, cr in assignments:
    status, _ = put(
        f'{BASE}/principal-roles/{pr}/catalog-roles/{catalog}',
        TOKEN,
        {'catalogRole': {'name': cr}}
    )
    print(f'  {"✅" if status in [200,201] else "⚠️ "} {pr} → {cr} on {catalog} ({status})')

print('\n✅ Complex RBAC setup complete!')
print('\n📊 Summary:')
print('  Principals:      alice, bob, carol, trino-service, etl-service')
print('  Principal Roles: data-engineer, data-analyst, ml-engineer, ops-admin')
print('  Catalogs:        analytics-catalog, ml-catalog')
print('  alice has dual roles: data-engineer + ops-admin')

✅ Connected

📋 Creating Principal Roles...
  ⚠️  principal-role: data-engineer (409)
  ⚠️  principal-role: data-analyst (409)
  ⚠️  principal-role: ml-engineer (409)
  ⚠️  principal-role: ops-admin (409)

👤 Creating Principals...
  ⚠️  principal: alice (409) — {'error': {'message': 'Cannot create Principal alice. Principal already exists or resolution failed', 'type': 'AlreadyExistsException', 'code': 409}}
  ⚠️  principal: bob (409) — {'error': {'message': 'Cannot create Principal bob. Principal already exists or resolution failed', 'type': 'AlreadyExistsException', 'code': 409}}
  ⚠️  principal: carol (409) — {'error': {'message': 'Cannot create Principal carol. Principal already exists or resolution failed', 'type': 'AlreadyExistsException', 'code': 409}}
  ⚠️  principal: trino-service (409) — {'error': {'message': 'Cannot create Principal trino-service. Principal already exists or resolution failed', 'type': 'AlreadyExistsException', 'code': 409}}
  ⚠️  principal: etl-service (409)

In [2]:
# Step 8 — Create Namespaces
print('\n📁 Creating Namespaces...')

# Need spark-service token for catalog API
def get_catalog_token(client_id, client_secret):
    r = requests.post(
        f'{POLARIS_URL}/api/catalog/v1/oauth/tokens',
        headers={'Polaris-Realm': REALM},
        data={
            'grant_type': 'client_credentials',
            'client_id': client_id,
            'client_secret': client_secret,
            'scope': 'PRINCIPAL_ROLE:ALL'
        }
    )
    return r.json()['access_token']

def catalog_headers(token):
    return {
        'Authorization': f'Bearer {token}',
        'Polaris-Realm': REALM,
        'Content-Type': 'application/json'
    }

# Use root token for catalog API (assigned ops-admin → admin-role on both catalogs)
CATALOG_TOKEN = TOKEN

namespaces = [
    # analytics-catalog namespaces
    ('analytics-catalog', 'raw',        's3a://data-catalog-bucket/analytics/raw'),
    ('analytics-catalog', 'processed',  's3a://data-catalog-bucket/analytics/processed'),
    ('analytics-catalog', 'reporting',  's3a://data-catalog-bucket/analytics/reporting'),
    # ml-catalog namespaces
    ('ml-catalog',        'features',   's3a://data-catalog-bucket/ml/features'),
    ('ml-catalog',        'models',     's3a://data-catalog-bucket/ml/models'),
    ('ml-catalog',        'experiments','s3a://data-catalog-bucket/ml/experiments'),
]

for catalog, ns, location in namespaces:
    r = requests.post(
        f'{POLARIS_URL}/api/catalog/v1/{catalog}/namespaces',
        headers=catalog_headers(CATALOG_TOKEN),
        json={
            'namespace': [ns],
            'properties': {'location': location}
        }
    )
    print(f'  {"✅" if r.status_code in [200,201] else "⚠️ "} {catalog}.{ns} ({r.status_code})')

# Step 9 — Create Tables
print('\n📊 Creating Tables...')
tables = [
    # analytics-catalog.raw
    ('analytics-catalog', 'raw', 'events', [
        {'id': 1, 'name': 'event_id',   'required': True,  'type': 'long'},
        {'id': 2, 'name': 'event_type', 'required': False, 'type': 'string'},
        {'id': 3, 'name': 'user_id',    'required': False, 'type': 'long'},
        {'id': 4, 'name': 'ts',         'required': False, 'type': 'timestamp'},
    ]),
    ('analytics-catalog', 'raw', 'users', [
        {'id': 1, 'name': 'user_id',    'required': True,  'type': 'long'},
        {'id': 2, 'name': 'username',   'required': False, 'type': 'string'},
        {'id': 3, 'name': 'email',      'required': False, 'type': 'string'},
        {'id': 4, 'name': 'created_at', 'required': False, 'type': 'timestamp'},
    ]),
    # analytics-catalog.processed
    ('analytics-catalog', 'processed', 'daily_active_users', [
        {'id': 1, 'name': 'date',       'required': True,  'type': 'date'},
        {'id': 2, 'name': 'dau_count',  'required': False, 'type': 'long'},
        {'id': 3, 'name': 'region',     'required': False, 'type': 'string'},
    ]),
    # analytics-catalog.reporting
    ('analytics-catalog', 'reporting', 'monthly_revenue', [
        {'id': 1, 'name': 'month',      'required': True,  'type': 'string'},
        {'id': 2, 'name': 'revenue',    'required': False, 'type': 'double'},
        {'id': 3, 'name': 'currency',   'required': False, 'type': 'string'},
    ]),
    # ml-catalog.features
    ('ml-catalog', 'features', 'user_features', [
        {'id': 1, 'name': 'user_id',    'required': True,  'type': 'long'},
        {'id': 2, 'name': 'feature_v',  'required': False, 'type': 'string'},
        {'id': 3, 'name': 'updated_at', 'required': False, 'type': 'timestamp'},
    ]),
    # ml-catalog.experiments
    ('ml-catalog', 'experiments', 'experiment_runs', [
        {'id': 1, 'name': 'run_id',     'required': True,  'type': 'string'},
        {'id': 2, 'name': 'model_name', 'required': False, 'type': 'string'},
        {'id': 3, 'name': 'accuracy',   'required': False, 'type': 'double'},
        {'id': 4, 'name': 'created_at', 'required': False, 'type': 'timestamp'},
    ]),
]

for catalog, ns, tname, fields in tables:
    r = requests.post(
        f'{POLARIS_URL}/api/catalog/v1/{catalog}/namespaces/{ns}/tables',
        headers=catalog_headers(CATALOG_TOKEN),
        json={
            'name': tname,
            'schema': {
                'type': 'struct',
                'schema-id': 0,
                'fields': fields
            },
            'partition-spec': {'spec-id': 0, 'fields': []},
            'write-order': {'order-id': 0, 'fields': []},
            'stage-create': False,
            'properties': {'write.format.default': 'parquet'}
        }
    )
    print(f'  {"✅" if r.status_code in [200,201] else "⚠️ "} {catalog}.{ns}.{tname} ({r.status_code})')

# Step 10 — Create Views
print('\n👁️  Creating Views...')
views = [
    # View on analytics-catalog.processed.daily_active_users
    (
        'analytics-catalog',
        'reporting',
        'weekly_active_users',
        'SELECT date, SUM(dau_count) as wau_count, region FROM processed.daily_active_users GROUP BY date, region',
        [
            {'id': 1, 'name': 'date',      'required': True,  'type': 'date'},
            {'id': 2, 'name': 'wau_count', 'required': False, 'type': 'long'},
            {'id': 3, 'name': 'region',    'required': False, 'type': 'string'},
        ]
    ),
    # View on analytics-catalog.raw.events
    (
        'analytics-catalog',
        'reporting',
        'recent_events',
        'SELECT event_id, event_type, user_id, ts FROM raw.events ORDER BY ts DESC LIMIT 1000',
        [
            {'id': 1, 'name': 'event_id',   'required': True,  'type': 'long'},
            {'id': 2, 'name': 'event_type', 'required': False, 'type': 'string'},
            {'id': 3, 'name': 'user_id',    'required': False, 'type': 'long'},
            {'id': 4, 'name': 'ts',         'required': False, 'type': 'timestamp'},
        ]
    ),
]

for catalog, ns, vname, sql, fields in views:
    r = requests.post(
        f'{POLARIS_URL}/api/catalog/v1/{catalog}/namespaces/{ns}/views',
        headers=catalog_headers(CATALOG_TOKEN),
        json={
            'name': vname,
            'schema': {
                'type': 'struct',
                'schema-id': 0,
                'fields': fields
            },
            'view-version': {
                'version-id': 1,
                'schema-id': 0,
                'timestamp-ms': 0,
                'summary': {'engine-name': 'spark'},
                'representations': [{
                    'type': 'sql',
                    'sql': sql,
                    'dialect': 'spark'
                }]
            },
            'properties': {}
        }
    )
    print(f'  {"✅" if r.status_code in [200,201] else "⚠️ "} {catalog}.{ns}.{vname} ({r.status_code})')
    if r.status_code not in [200, 201]:
        print(f'     {r.text[:100]}')

# Step 11 — Create Policies
print('\n📜 Creating Policies...')

# Check if policy API is available
r = requests.get(
    f'{POLARIS_URL}/api/management/v1/catalogs/analytics-catalog/policies',
    headers=h(TOKEN)
)

if r.status_code == 404:
    print('  ⚠️  Policy API not available in this Polaris version')
    print('     Policies require Polaris 1.4.0+')
    print('     Skipping policy creation')
else:
    policies = [
        {
            'catalog': 'analytics-catalog',
            'name': 'raw-data-retention',
            'type': 'data-compaction',
            'description': 'Compact raw data files every 7 days',
            'content': json.dumps({
                'enable': True,
                'target-file-size-bytes': 134217728,
                'strategy': 'binpack'
            })
        },
        {
            'catalog': 'analytics-catalog',
            'name': 'snapshot-expiry',
            'type': 'snapshot-expiry',
            'description': 'Expire snapshots older than 30 days',
            'content': json.dumps({
                'min-snapshots-to-keep': 5,
                'max-snapshot-age-ms': 2592000000
            })
        },
    ]
    for p in policies:
        r = requests.post(
            f'{POLARIS_URL}/api/management/v1/catalogs/{p["catalog"]}/policies',
            headers=h(TOKEN),
            json={
                'policy': {
                    'name': p['name'],
                    'policyType': p['type'],
                    'description': p['description'],
                    'content': p['content']
                }
            }
        )
        print(f'  {"✅" if r.status_code in [200,201] else "⚠️ "} {p["catalog"]}/{p["name"]} ({r.status_code})')
        if r.status_code not in [200, 201]:
            print(f'     {r.text[:100]}')

print('\n✅ Full RBAC setup complete!')
print('\n📊 Final Summary:')
print('  Principals:      alice, bob, carol, trino-service, etl-service')
print('  Principal Roles: data-engineer, data-analyst, ml-engineer, ops-admin')
print('  Catalogs:        analytics-catalog, ml-catalog')
print('  Namespaces:      raw, processed, reporting | features, models, experiments')
print('  Tables:          events, users, daily_active_users, monthly_revenue,')
print('                   user_features, experiment_runs')
print('  Views:           weekly_active_users, recent_events')
print('  alice dual role: data-engineer + ops-admin (admin on both catalogs)')


📁 Creating Namespaces...
  ⚠️  analytics-catalog.raw (409)
  ⚠️  analytics-catalog.processed (409)
  ⚠️  analytics-catalog.reporting (409)
  ⚠️  ml-catalog.features (409)
  ⚠️  ml-catalog.models (409)
  ⚠️  ml-catalog.experiments (409)

📊 Creating Tables...
  ⚠️  analytics-catalog.raw.events (409)
  ⚠️  analytics-catalog.raw.users (409)
  ⚠️  analytics-catalog.processed.daily_active_users (409)
  ⚠️  analytics-catalog.reporting.monthly_revenue (409)
  ⚠️  ml-catalog.features.user_features (409)
  ⚠️  ml-catalog.experiments.experiment_runs (409)

👁️  Creating Views...
  ⚠️  analytics-catalog.reporting.weekly_active_users (400)
     {"error":{"message":"Cannot parse missing field: default-namespace","type":"IllegalArgumentException
  ⚠️  analytics-catalog.reporting.recent_events (400)
     {"error":{"message":"Cannot parse missing field: default-namespace","type":"IllegalArgumentException

📜 Creating Policies...
  ⚠️  Policy API not available in this Polaris version
     Policies requir

In [3]:
# Recreate view
r = requests.post(
    f'{POLARIS_URL}/api/catalog/v1/analytics-catalog/namespaces/reporting/views',
    headers=h(TOKEN),
    json={
        'name': 'weekly_active_users',
        'schema': {
            'type': 'struct',
            'schema-id': 0,
            'fields': [
                {'id': 1, 'name': 'date',      'required': True,  'type': 'date'},
                {'id': 2, 'name': 'wau_count', 'required': False, 'type': 'long'},
                {'id': 3, 'name': 'region',    'required': False, 'type': 'string'},
            ]
        },
        'view-version': {
            'version-id': 1,
            'schema-id': 0,
            'timestamp-ms': 0,
            'summary': {'engine-name': 'spark'},
            'default-namespace': ['reporting'],      # ← required field
            'representations': [{
                'type': 'sql',
                'sql': 'SELECT date, SUM(dau_count) as wau_count, region FROM processed.daily_active_users GROUP BY date, region',
                'dialect': 'spark'
            }]
        },
        'properties': {}
    }
)
print(f'View create: {r.status_code}')
if r.text:
    print(r.text[:300])

# grant
fine_grained_grants = [
    # Table grant — use 'tableName'
    ('analytics-catalog', 'analyst-role', {
        'grant': {
            'type': 'table',
            'namespace': ['raw'],
            'tableName': 'events',
            'privilege': 'TABLE_READ_DATA'
        }
    }),
    # View grant — likely 'viewName'
    ('analytics-catalog', 'analyst-role', {
        'grant': {
            'type': 'view',
            'namespace': ['reporting'],
            'viewName': 'weekly_active_users',
            'privilege': 'VIEW_READ_PROPERTIES'
        }
    }),
    ('ml-catalog', 'ml-role', {
        'grant': {
            'type': 'table',
            'namespace': ['features'],
            'tableName': 'user_features',
            'privilege': 'TABLE_WRITE_DATA'
        }
    }),
]

print('⚡ Granting fine-grained privileges...')
for cname, role, body in fine_grained_grants:
    r = requests.put(
        f'{POLARIS_URL}/api/management/v1/catalogs/{cname}/catalog-roles/{role}/grants',
        headers=h(TOKEN),
        json=body
    )
    grant = body['grant']
    print(f'  {"✅" if r.status_code in [200,201] else "⚠️ "} {r.status_code} — {grant["type"]}:{grant.get("tableName") or grant.get("viewName","*")}')
    if r.status_code not in [200, 201] and r.text:
        print(f'     {r.text[:200]}')

View create: 409
{"error":{"message":"View already exists: reporting.weekly_active_users","type":"AlreadyExistsException","code":409}}
⚡ Granting fine-grained privileges...
  ⚠️  500 — table:events
     {"error":{"message":"Failed to write to grant records due to Failed due to 'ERROR: duplicate key value violates unique constraint \"grant_records_pkey\"\n  Detail: Key (realm_id, securable_catalog_id,
  ⚠️  500 — view:weekly_active_users
     {"error":{"message":"Failed to write to grant records due to Failed due to 'ERROR: duplicate key value violates unique constraint \"grant_records_pkey\"\n  Detail: Key (realm_id, securable_catalog_id,
  ⚠️  500 — table:user_features
     {"error":{"message":"Failed to write to grant records due to Failed due to 'ERROR: duplicate key value violates unique constraint \"grant_records_pkey\"\n  Detail: Key (realm_id, securable_catalog_id,
